# Merchant Registration Batch

`merchant_info`의 상가 정보 CSV에서 `상호명`을 읽어 SSAFY 금융 API `createMerchant`를 호출하고, 응답의 `merchantId`, `merchantName`을 CSV로 저장합니다.

- 고정 `categoryId`: `CG-9ca85f66311a23d`
- 성공 결과 CSV: `merchantId`, `merchantName`
- 실패 결과 CSV: `merchantName`, `responseCode`, `responseMessage`, `errorType`


In [ ]:
import os
import sys
from datetime import datetime
from pathlib import Path
from pprint import pprint

import httpx
from dotenv import load_dotenv

current = Path.cwd().resolve()
candidates = [current, *current.parents]
PROJECT_ROOT = next(
    (
        path
        for path in candidates
        if (path / 'lab').exists() and (path / 'pyproject.toml').exists()
    ),
    current,
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / '.env')

from lab.finapi.merchant_registration import (
    CREATE_MERCHANT_URL,
    DEFAULT_CATEGORY_ID,
    discover_merchant_csv_paths,
    load_merchant_names,
    register_merchants,
    write_failure_rows,
    write_success_rows,
)

INPUT_PATH = PROJECT_ROOT / 'lab' / 'merchant_info'
OUTPUT_DIR = PROJECT_ROOT / 'lab' / 'finapi' / 'outputs'
API_KEY = os.getenv('SSAFY_FIN_API_KEY', '')
INSTITUTION_CODE = os.getenv('SSAFY_FIN_INSTITUTION_CODE', '00100')
FINTECH_APP_NO = os.getenv('SSAFY_FIN_FINTECH_APP_NO', '001')
CATEGORY_ID = DEFAULT_CATEGORY_ID
BASE_URL = CREATE_MERCHANT_URL
MAX_MERCHANTS = None

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'INPUT_PATH: {INPUT_PATH}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')
print(f'CATEGORY_ID: {CATEGORY_ID}')
print('API_KEY configured:', bool(API_KEY))


In [ ]:
csv_paths = discover_merchant_csv_paths(INPUT_PATH)
merchant_names = load_merchant_names(csv_paths)

if MAX_MERCHANTS is not None:
    merchant_names = merchant_names[:MAX_MERCHANTS]

print(f'CSV files: {len(csv_paths)}')
print(f'Merchant names: {len(merchant_names)}')
print('Sample merchant names:')
pprint(merchant_names[:10])


In [ ]:
if not API_KEY:
    raise ValueError('SSAFY_FIN_API_KEY 환경 변수가 비어 있습니다.')

run_at = datetime.now().strftime('%Y%m%d_%H%M%S')
success_csv = OUTPUT_DIR / f'registered_merchants_{run_at}.csv'
failure_csv = OUTPUT_DIR / f'merchant_registration_failures_{run_at}.csv'

with httpx.Client() as client:
    success_rows, failure_rows = register_merchants(
        merchant_names=merchant_names,
        api_key=API_KEY,
        client=client,
        category_id=CATEGORY_ID,
        base_url=BASE_URL,
        institution_code=INSTITUTION_CODE,
        fintech_app_no=FINTECH_APP_NO,
    )

write_success_rows(success_csv, success_rows)
write_failure_rows(failure_csv, failure_rows)

print(f'Success rows: {len(success_rows)} -> {success_csv}')
print(f'Failure rows: {len(failure_rows)} -> {failure_csv}')


In [ ]:
print('Success preview:')
pprint(success_rows[:5])

print('\nFailure preview:')
pprint(failure_rows[:5])
